# 00 — Colab setup

Run this first in every new Colab session (select a Colab runtime as the kernel in VS Code).

It will:
1. show the runtime (Python, GPU);
2. clone or update the code from GitHub and install it;
3. check that the package works by running the tests.

**Storage (plan B):** your PC is the permanent home of data and results. Colab is only for GPU compute, and its disk (`/content/dtwin`) is **wiped when the runtime disconnects**.
- To send files from the PC to Colab: right-click them in the VS Code Explorer and choose **Upload to Colab**. They arrive in `/content`.
- Anything Colab produces that is worth keeping must be brought back to the PC. Phase notebooks will say how.

In [ ]:
import sys, platform, subprocess, shutil, os
print("Python:", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):  # absent on CPU runtimes
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
else:
    gpu = ""
print("GPU:", gpu or "none (CPU runtime) - fine for setup; use a GPU runtime for training phases")

In [ ]:
# Clone (first time) or update (later sessions) the public repository.
REPO_URL = "https://github.com/plufle/Patient-adaptive-ECG-PPG-hypotension-warning-with-competing-explanation-search.git"
REPO_DIR = "/content/repo"
BRANCH = "main"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Install the package. EXTRAS grows as phases need more (e.g. "dev,sim" in P1, "dev,sim,ml" in P6).
EXTRAS = "dev"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# Make the editable install visible to this already-running kernel.
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)

import dtwin
from dtwin.utils.io import get_paths, on_colab
paths = get_paths(create=True)
print("dtwin", dtwin.__version__, "| on_colab:", on_colab())
print("storage root:", paths.root, "| persistent:", paths.persistent)

In [ ]:
# Run the test suite on the Colab runtime.
result = subprocess.run([sys.executable, "-m", "pytest", "-q", os.path.join(REPO_DIR, "tests")],
                        capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout[-3000:], result.stderr[-2000:])
assert result.returncode == 0, "tests failed"